# Preparacion del dataset para ML

Mezclaremos tabla de readmisiones(del notebook 02) con tabla de triage(signos vitales al llegar a urgencias)

In [1]:
import pandas as pd
import numpy as np
import os

data_path = r"mimic-iv-ed-2.2/ed"

# Tabla base: estancias con target de readmision
df_edstays = pd.read_csv("edstays_with_target.csv", parse_dates=["intime", "outtime"])

# Signos vitales en triage
df_triage = pd.read_csv(os.path.join(data_path, "triage.csv.gz"))

# Diagn�sticos

print(f"edstays:   {df_edstays.shape}")
print(f"triage:    {df_triage.shape}")

edstays:   (425087, 12)
triage:    (425087, 11)


## 1. Primera visita por paciente

Nos quedamos solo con la **primera visita** de cada paciente porque queremos predecir si un paciente volver tras su primera visita a urgencias

In [2]:
df_first = (
    df_edstays
    .sort_values("intime")
    .groupby("subject_id")
    .first()
    .reset_index()
)

print(f"Pacientes unicos: {len(df_first):,}")
print(f"Readmitidos (1):  {df_first['readmitted'].sum():,}  ({df_first['readmitted'].mean()*100:.1f}%)")
print(f"No readmitidos (0): {(df_first['readmitted']==0).sum():,}  ({(df_first['readmitted']==0).mean()*100:.1f}%)")

Pacientes unicos: 205,504
Readmitidos (1):  17,362  (8.4%)
No readmitidos (0): 188,142  (91.6%)


## 2. Merge con triage (signos vitales)

El triage recoge los signos vitales al llegar a urgencias: temperatura, frecuencia cardiaca, presion arterial, saturacion de oxigeno, etc.

Los nulos los imputamos con valores fisiologicos normales (no con la media del dataset) porque representan casos donde no se midio el signo vital, y asumir normalidad es clinicamente razonable.

In [3]:
# Imputar nulos de triage con valores fisiol�gicos normales
valores_normales = {
    "temperature": 36.8,
    "heartrate":   75,
    "resprate":    16,
    "o2sat":       98,
    "sbp":         120,
    "dbp":         80,
    "pain":        0,
    "acuity":      5
}

print("Nulos en triage antes de imputar:")
print(df_triage[list(valores_normales.keys())].isnull().sum())

df_triage_filled = df_triage.copy()
for col, val in valores_normales.items():
    df_triage_filled[col] = df_triage_filled[col].fillna(val)

# Merge: primera visita + triage (quitamos columnas redundantes o de texto libre)
df = df_first.merge(
    df_triage_filled.drop(columns=["subject_id", "chiefcomplaint"], errors="ignore"),
    on="stay_id",
    how="left"
)

print(f"Shape tras merge con triage: {df.shape}")

Nulos en triage antes de imputar:
temperature    23415
heartrate      17090
resprate       20353
o2sat          20596
sbp            18291
dbp            19091
pain           12933
acuity          6987
dtype: int64
Shape tras merge con triage: (205504, 20)


## 4. Nuevas variables: ingreso y tiempo en urgencias

Creamos dos variables nuevas:

- **ingreso**: si el paciente fue ingresado al hospital desde urgencias (1) o no (0). Se detecta por la presencia de .
- **intime_min_dia**: hora de llegada a urgencias en minutos del d�a (0-1440). Captura si la visita fue de noche, de madrugada, etc.
- **ed_los_min**: tiempo total de estancia en urgencias en minutos.

In [5]:
# Ingreso al hospital (1 si tiene hadm_id, 0 si no)
df["ingreso"] = df["hadm_id"].notna().astype(int)

# Hora de llegada en minutos del d�a
df["intime_min_dia"] = df["intime"].dt.hour * 60 + df["intime"].dt.minute

# Duraci�n de estancia en urgencias (minutos)
df["ed_los_min"] = (df["outtime"] - df["intime"]).dt.total_seconds() / 60

print(df[["ingreso", "intime_min_dia", "ed_los_min"]].describe().round(1))

        ingreso  intime_min_dia  ed_los_min
count  205504.0        205504.0    205504.0
mean        0.5           842.2       390.7
std         0.5           363.3       363.0
min         0.0             0.0     -1364.0
25%         0.0           635.0       193.0
50%         1.0           882.0       297.0
75%         1.0          1126.0       453.0
max         1.0          1439.0     20462.4


In [7]:
# Eliminar filas con ed_los_min negativo (error en los datos: outtime < intime)
negativos = (df["ed_los_min"] < 0).sum()
print(f"Filas con ed_los_min negativo (se eliminan): {negativos}")
df = df[df["ed_los_min"] >= 0].copy()

Filas con ed_los_min negativo (se eliminan): 4


## 5. Codificacion de variables categoricas y guardado

Los modelos de ML necesitan variables numericas. Codificamos las categoricas con un mapeo ordinal simple.

Eliminamos columnas que no usaremos en el modelo (identificadores, fechas, texto libre, columnas redundantes).

In [8]:
df_ml = df.copy()

# Convertir pain a numerico (a veces viene como string)
df_ml["pain"] = pd.to_numeric(df_ml["pain"], errors="coerce").fillna(0).astype(int)

# Codificacion de categoricas
map_gender      = {"M": 0, "F": 1}
map_arrival     = {cat: i for i, cat in enumerate(sorted(df_ml["arrival_transport"].dropna().unique()))}
map_disposition = {cat: i for i, cat in enumerate(sorted(df_ml["disposition"].dropna().unique()))}
map_race        = {cat: i for i, cat in enumerate(sorted(df_ml["race"].dropna().unique()))}

df_ml["gender_num"]           = df_ml["gender"].map(map_gender)
df_ml["arrival_transport_num"] = df_ml["arrival_transport"].map(map_arrival)
df_ml["disposition_num"]       = df_ml["disposition"].map(map_disposition)
df_ml["race_num"]              = df_ml["race"].map(map_race)

print("disposition:", map_disposition)
print("arrival_transport:", map_arrival)

# Eliminar columnas que no van al modelo
cols_drop = [
    "subject_id", "hadm_id", "intime", "outtime",
    "next_intime", "days_to_next_visit",
    "gender", "arrival_transport", "disposition", "race",
    "icd_code", "icd_version", "icd_title"
]
df_ml = df_ml.drop(columns=cols_drop, errors="ignore")

print(f"Shape final: {df_ml.shape}")
print(f"Nulos restantes:{df_ml.isnull().sum()[df_ml.isnull().sum()>0]}")
df_ml.head()

disposition: {'ADMITTED': 0, 'ELOPED': 1, 'EXPIRED': 2, 'HOME': 3, 'LEFT AGAINST MEDICAL ADVICE': 4, 'LEFT WITHOUT BEING SEEN': 5, 'OTHER': 6, 'TRANSFER': 7}
arrival_transport: {'AMBULANCE': 0, 'HELICOPTER': 1, 'OTHER': 2, 'UNKNOWN': 3, 'WALK IN': 4}
Shape final: (205500, 17)
Nulos restantes:Series([], dtype: int64)


,stay_id,readmitted,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,ingreso,intime_min_dia,ed_los_min,gender_num,arrival_transport_num,disposition_num,race_num
0,33258284,0,98.40,70.0,16.0,97.0,106.0,63.0,0,3.0,1,1157,253.0,1,0,0,28
1,35203156,0,97.50,78.0,16.0,100.0,114.0,71.0,0,2.0,1,1236,404.0,0,4,0,28
2,32522732,1,98.21,83.0,20.0,100.0,112.0,81.0,5,3.0,1,994,99.0,0,4,3,28
3,38081480,1,97.60,81.0,18.0,99.0,120.0,71.0,5,3.0,0,124,235.0,0,4,3,28
4,32642808,0,97.60,81.0,16.0,100.0,148.0,83.0,0,3.0,1,1311,255.7,1,4,0,28


In [9]:
df_ml.to_csv("datos_edstays_triage.csv", index=False, encoding="utf-8")
print("Guardado: datos_edstays_triage.csv")
print(f"Filas: {len(df_ml):,}  |  Columnas: {df_ml.shape[1]}")
print(f"Columnas: {df_ml.columns.tolist()}")

Guardado: datos_edstays_triage.csv
Filas: 205,500  |  Columnas: 17
Columnas: ['stay_id', 'readmitted', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'ingreso', 'intime_min_dia', 'ed_los_min', 'gender_num', 'arrival_transport_num', 'disposition_num', 'race_num']
